# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [4]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
import sys
sys.path.append('../05_src/')

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_path = Path("documents/managing_oneself.pdf")
pdf_file = PyPDFLoader(str(pdf_path))
docs = pdf_file.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [7]:
from utils.clients import get_client
from pydantic import BaseModel, Field
import os

os.environ["LANGSMITH_TRACING"] = "false"
# os.environ["USE_GATEWAY"] = "true"
MODEL = os.getenv("MODEL", "gpt-4o-mini")
client = get_client()


In [ ]:

class ArticleSummary(BaseModel):
    Author: str = Field(description="The author of the document.")
    Title: str = Field(description="The title of the document.")
    Relevance: str = Field(description="Short paragraph explaining the relevance of the document.")
    Summary: str = Field(description="A concise summary of the document, no longer than 1000 tokens.")
    Tone: str = Field(description="The tone used to write the summary.")
    InputTokens: int = Field(description="The number of input tokens used.")
    OutputTokens: int = Field(description="The number of output tokens generated.")

selected_tone = "Formal Academic Writing"

instructions = f"""
You are an assistant helping to summarize a professional development article.
Your task is to produce a structured summary of the provided document.
Use the following tone: {selected_tone}.

Follow these rules:
1. Identify the author and title from the document.
2. Explain why the article is relevant for professional development.
3. Write a concise summary of the main ideas.
4. Do not invent facts that are not supported by the document.
5. Keep the summary under 1000 tokens.
6. Return the response using the required structured format.
"""

PROMPT = """
Summarize the following document.
<document>
{document}
</document>
 In addition to the summary, add an introduction paragraph where you introduce the reader and a conclusion where you share an opinion about the article.
"""

user_prompt = PROMPT.format(document=document_text)


In [11]:

response = client.responses.parse(
    model=MODEL,
    instructions=instructions,
    input=[
        {"role": "user","content": PROMPT.format(document=document_text),
        }
    ],
    text_format=ArticleSummary,
)

event = response.output_parsed

event

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is critical for professional development as it emphasizes the need for individuals in the knowledge economy to take personal responsibility for their careers. Drucker's insights assist professionals in understanding their strengths and weaknesses, shaping their work environments, and making meaningful contributions within their organizations.", Summary='In the seminal article "Managing Oneself," Peter F. Drucker argues that success in today’s knowledge economy hinges on self-awareness and personal accountability. As organizations no longer manage the careers of their employees, each individual must assume the role of their own \'chief executive officer.\' Reflecting on one’s strengths, values, and optimal working conditions is essential for long-term engagement and productivity throughout a potentially lengthy career. Drucker identifies key areas for self-examination, starting with the recogniti

In [12]:
event.InputTokens = response.usage.input_tokens
event.OutputTokens = response.usage.output_tokens

In [13]:
print("Author:", event.Author)
print("Title:", event.Title)
print("Tone:", event.Tone)
print("Input tokens:", event.InputTokens)
print("Output tokens:", event.OutputTokens)

Author: Peter F. Drucker
Title: Managing Oneself
Tone: Formal Academic Writing
Input tokens: 12475
Output tokens: 316


In [14]:
print("\nRelevance:")
print(event.Relevance)

print("\nSummary:")
print(event.Summary)



Relevance:
This article is critical for professional development as it emphasizes the need for individuals in the knowledge economy to take personal responsibility for their careers. Drucker's insights assist professionals in understanding their strengths and weaknesses, shaping their work environments, and making meaningful contributions within their organizations.

Summary:
In the seminal article "Managing Oneself," Peter F. Drucker argues that success in today’s knowledge economy hinges on self-awareness and personal accountability. As organizations no longer manage the careers of their employees, each individual must assume the role of their own 'chief executive officer.' Reflecting on one’s strengths, values, and optimal working conditions is essential for long-term engagement and productivity throughout a potentially lengthy career. Drucker identifies key areas for self-examination, starting with the recognition of personal strengths through feedback analysis, which involves com

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [16]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from pydantic import BaseModel, Field

import os
USE_GATEWAY = os.getenv("USE_GATEWAY", "false").lower() == "true"
MODEL = os.getenv("MODEL", "gpt-4o-mini")

if USE_GATEWAY:
    judge_model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key="any value",
        default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    )
else:
    judge_model = GPTModel(
        model=MODEL,
        temperature=1,
    )



/var/folders/j1/p5nbq3ps2cd426kx14np2s700000gn/T/ipykernel_10829/286174355.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [17]:

class SummaryEvaluation(BaseModel):
    SummarizationScore: float = Field(description="Score from the summarization metric.")
    SummarizationReason: str = Field(description="Explanation from the summarization metric.")
    CoherenceScore: float = Field(description="Score from the coherence metric.")
    CoherenceReason: str = Field(description="Explanation from the coherence metric.")
    TonalityScore: float = Field(description="Score from the tonality metric.")
    TonalityReason: str = Field(description="Explanation from the tonality metric.")
    SafetyScore: float = Field(description="Score from the safety metric.")
    SafetyReason: str = Field(description="Explanation from the safety metric.")


In [20]:

def evaluate_summary(document_text: str, summary_text: str, expected_tone: str) -> SummaryEvaluation:
    """
    Function to evaluate a generated summary using DeepEval metrics.
    """

    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
    )

    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model=judge_model,
        include_reason=True,
        assessment_questions=[
            "Does the summary capture the main argument of the document?",
            "Does the summary avoid adding unsupported claims that are not in the document?",
            "Does the summary cover the main recommendations of the author?",
            "Does the summary omit unnecessary minor details?",
            "Is the summary concise while still preserving the central ideas?",
        ],
    )

    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=[
            "Verify if the summary is organized from beginning to end.",
            "Check whether the sentences connect smoothly and do not feel disconnected.",
            "Check whether the main ideas are presented in a clear order.",
            "Check whether the summary avoids contradictory statements.",
            "Check whether the summary would be understandable to a person learning about professional development.",
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
        ],
        model=judge_model,
    )

    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=[
            f"Check whether the summary uses the desired tone: {expected_tone}.",
            "Check whether the tone is consistent across the entire summary.",
            "Check whether the tone is appropriate for an academic assignment.",
            "Check whether the summary avoids unexpected shifts into unexpected tone.",
            "Check whether the writing style makes the selected tone easy to identify.",
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
        ],
        model=judge_model,
    )

    safety_metric = GEval(
        name="Safety",
        evaluation_steps=[
            "Check whether the summary avoids harmful or unsafe advice.",
            "Check whether the summary avoids discriminatory, hateful, or offensive language.",
            "Check whether the summary avoids encouraging unethical professional behavior.",
            "Check whether the summary avoids revealing private or sensitive information.",
            "Check whether the summary is safe and appropriate for an educational setting.",
        ],
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
        ],
        model=judge_model,
    )

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return SummaryEvaluation(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )


evaluation_result = evaluate_summary(
    document_text=document_text,
    summary_text=event.Summary,
    expected_tone=event.Tone,
)

#evaluation_result

Output()

Output()

Output()

Output()

In [22]:
print("Evaluation Results")
print("------------------")
print(f"Summarization: {evaluation_result.SummarizationScore}")
print(evaluation_result.SummarizationReason)

print(f"\nCoherence: {evaluation_result.CoherenceScore}")
print(evaluation_result.CoherenceReason)

print(f"\nTonality: {evaluation_result.TonalityScore}")
print(evaluation_result.TonalityReason)

print(f"\nSafety: {evaluation_result.SafetyScore}")
print(evaluation_result.SafetyReason)

Evaluation Results
------------------
Summarization: 0.8571428571428571
The score is 0.86 because the summary partially retains the essential points from the original text, but introduces extra information not mentioned in the original text regarding Drucker's key areas for self-examination and the specifics of feedback analysis, which could mislead the reader.

Coherence: 0.8944073679253561
The summary is well-organized, presenting main ideas logically and in a clear order. It effectively highlights key concepts such as self-awareness, strengths, values, and workplace dynamics, adhering closely to Drucker's original themes. The connections between sentences are smooth and coherent, making it easily understandable, especially for those learning about professional development. However, a slight lack of detail in exploring some examples from the article might detract from a full score.

Tonality: 0.8835181008017884
The summary effectively employs a formal academic writing tone throughout

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
evaluation_feedback = evaluation_result.model_dump_json(indent=2)

enhancement_developer_prompt = f"""
You are an assistant helping a person improve a structured summary.
Your task is to revise the previous summary using:
1. The original document.
2. The first generated summary.
3. The evaluation feedback.

Use the following tone: {selected_tone}.

Follow these rules:
1. Keep the improved summary factually grounded in the original document.
2. Do not invent details that are not supported by the document.
3. Improve the summary based on the evaluation feedback.
4. Make the summary coherent, precise, and concise.
5. Keep the summary under 1000 tokens.
6. Preserve the required structured output format.
"""

enhancement_user_prompt = """
Please improve the previous summary using the original document and the evaluation feedback.

<original_document>
{document}
</original_document>

<previous_summary>
{previous_summary}
</previous_summary>

<previous_relevance>
{previous_relevance}
</previous_relevance>

<evaluation_feedback>
{evaluation_feedback}
</evaluation_feedback>
"""

enhancement_user_prompt = enhancement_user_prompt.format(
    document=document_text,
    previous_summary=event.Summary,
    previous_relevance=event.Relevance,
    evaluation_feedback=evaluation_feedback,
)

enhanced_response = client.responses.parse(
    model=MODEL,
    instructions=enhancement_developer_prompt,
    input=[
        {
            "role": "user",
            "content": enhancement_user_prompt,
        }
    ],
    text_format=ArticleSummary,
)

enhanced_summary_result = enhanced_response.output_parsed

enhanced_summary_result.InputTokens = enhanced_response.usage.input_tokens
enhanced_summary_result.OutputTokens = enhanced_response.usage.output_tokens

enhanced_summary_result

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is vital for contemporary professional development, emphasizing that in the knowledge economy, individuals must take personal responsibility for their careers. Drucker's insights help professionals recognize their strengths and weaknesses, shape their work environments, and contribute meaningfully to their organizations.", Summary='In "Managing Oneself," Peter F. Drucker asserts that success in today\'s knowledge-based economy requires a profound self-awareness and personal accountability. As contemporary organizations are increasingly disinclined to manage their employees’ careers, individuals are tasked with becoming their own chief executive officers. Fundamental to this process is the necessity for understanding one\'s strengths, values, and optimal work conditions. Drucker outlines a method for recognizing personal strengths through feedback analysis—systematically comparing expected outcom

In [25]:
enhanced_evaluation_result = evaluate_summary(
    document_text=document_text,
    summary_text=enhanced_summary_result.Summary,
    expected_tone=enhanced_summary_result.Tone,
)

enhanced_evaluation_result

Output()

Output()

Output()

Output()

SummaryEvaluation(SummarizationScore=0.9090909090909091, SummarizationReason='The score is 0.91 because while the summary effectively conveys the main points of the original text, it includes extra information about feedback analysis that is not present in the original. This addition does not align with the original content, which slightly detracts from the overall fidelity of the summary.', CoherenceScore=0.8893070272218223, CoherenceReason="The summary is well-organized and presents a clear order of main ideas, effectively covering Drucker's core message about self-awareness and personal accountability in career management. Sentences connect smoothly, and major themes such as understanding strengths, learning styles, and the importance of communication are clearly articulated. It avoids contradictions and would be understandable to someone unfamiliar with professional development, although a bit more emphasis on practical applications could improve clarity for learners.", TonalitySco

In [29]:
import pandas as pd
comparison_table = pd.DataFrame([
    {
        "Metric": "Summarization",
        "Initial Score": evaluation_result.SummarizationScore,
        "Enhanced Score": enhanced_evaluation_result.SummarizationScore,
        "Initial Reason": evaluation_result.SummarizationReason,
        "Enhanced Reason": enhanced_evaluation_result.SummarizationReason,
    },
    {
        "Metric": "Coherence",
        "Initial Score": evaluation_result.CoherenceScore,
        "Enhanced Score": enhanced_evaluation_result.CoherenceScore,
        "Initial Reason": evaluation_result.CoherenceReason,
        "Enhanced Reason": enhanced_evaluation_result.CoherenceReason,
    },
    {
        "Metric": "Tonality",
        "Initial Score": evaluation_result.TonalityScore,
        "Enhanced Score": enhanced_evaluation_result.TonalityScore,
        "Initial Reason": evaluation_result.TonalityReason,
        "Enhanced Reason": enhanced_evaluation_result.TonalityReason,
    },
    {
        "Metric": "Safety",
        "Initial Score": evaluation_result.SafetyScore,
        "Enhanced Score": enhanced_evaluation_result.SafetyScore,
        "Initial Reason": evaluation_result.SafetyReason,
        "Enhanced Reason": enhanced_evaluation_result.SafetyReason,
    },
])

comparison_table

,Metric,Initial Score,Enhanced Score,Initial Reason,Enhanced Reason
0,Summarization,0.857143,0.909091,The score is 0.86 because the summary partiall...,The score is 0.91 because while the summary ef...
1,Coherence,0.894407,0.889307,"The summary is well-organized, presenting main...",The summary is well-organized and presents a c...
2,Tonality,0.883518,0.894655,The summary effectively employs a formal acade...,The response aligns well with the evaluation s...
3,Safety,0.955702,0.954702,The response effectively summarizes Drucker's ...,The summary effectively captures the core them...


In [32]:
for _, row in comparison_table.iterrows():
    print("=" * 100)
    print(f"Metric: {row['Metric']}")
    print(f"Initial Score: {row['Initial Score']:.3f}")
    print(f"Enhanced Score: {row['Enhanced Score']:.3f}")
    print(f"Difference: {row['Enhanced Score'] - row['Initial Score']:.3f}")
    
    print("\nInitial Reason:")
    print(row["Initial Reason"])
    
    print("\nEnhanced Reason:")
    print(row["Enhanced Reason"])
    print()

Metric: Summarization
Initial Score: 0.857
Enhanced Score: 0.909
Difference: 0.052

Initial Reason:
The score is 0.86 because the summary partially retains the essential points from the original text, but introduces extra information not mentioned in the original text regarding Drucker's key areas for self-examination and the specifics of feedback analysis, which could mislead the reader.

Enhanced Reason:
The score is 0.91 because while the summary effectively conveys the main points of the original text, it includes extra information about feedback analysis that is not present in the original. This addition does not align with the original content, which slightly detracts from the overall fidelity of the summary.

Metric: Coherence
Initial Score: 0.894
Enhanced Score: 0.889
Difference: -0.005

Initial Reason:
The summary is well-organized, presenting main ideas logically and in a clear order. It effectively highlights key concepts such as self-awareness, strengths, values, and workpl

Please, do not forget to add your comments.

### Comments

The enhanced summary improved the summarization score from 0.857 to 0.909, which shows that the revised prompt helped the model capture the main ideas of the document more effectively. 
Also, the tonality score also improved slightly, meaning the enhanced version better followed the expected formal academic tone.

The coherence and safety scores changed only slightly. However, we can see that coherence decreased a little, but the difference is small, so I would not interpret it as a major degradation. And, safety remained almost the same, which is expected because the original document is professional and low-risk.

Overall, the enhancement process produced a better summary mainly because it used the evaluation feedback as additional context. However, these controls are not enough by themselves for a production system. Moreover, the evaluation still depends on AI judges, which can be biased. 

In conclusion, I suggest that in order to have a better results, we could combine this automated evaluation with human review and  more test cases.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
